In [1]:

from google.colab import drive
import zipfile, os, glob

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from google.colab import drive
import zipfile, os, glob

drive.mount('/content/drive')

zip_path = '/content/drive/MyDrive/recognition.zip'
extract_path = '/content/data_csv'

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)

csv_file = glob.glob(extract_path + '/**/*.csv', recursive=True)[0]
print(" CSV Found:", csv_file)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 CSV Found: /content/data_csv/recognition/test.csv


In [ ]:

import torch
from torch.utils.data import Dataset
from PIL import Image
import pandas as pd

class OCRDataset(Dataset):
    def __init__(self, csv_file, base_dir, transform=None):
    
        #Loads image paths + text labels from CSV
        self.df = pd.read_csv(csv_file)
        self.base_dir = base_dir
        self.transform = transform

        # Build character-level vocab from dataset
        chars = set()
        for txt in self.df['Text'].astype(str):
            chars.update(list(txt))

        # Reserve indexes: 0 = <pad>, 1 = <sos>, 2 = <eos>
        self.vocab = ['<pad>', '<sos>', '<eos>'] + sorted(list(chars))
        self.char2idx = {c: i for i, c in enumerate(self.vocab)}
        self.idx2char = {i: c for i, c in enumerate(self.vocab)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        #Load image, convert to grayscale, apply transform and Convert text → sequence of vocabulary indexes
        row = self.df.iloc[idx]
        img_path = f"{self.base_dir}/{row['Filepath']}"
        image = Image.open(img_path).convert('L')  # grayscale
        if self.transform:
            image = self.transform(image)
        text = str(row['Text'])
        target = [self.char2idx['<sos>']] + [self.char2idx[c] for c in text] + [self.char2idx['<eos>']]
        return image, torch.tensor(target, dtype=torch.long)

#Makes batch padded properly for training
def collate_fn(batch):
    images, targets = zip(*batch)
    images = torch.stack(images)  

    lengths = [t.size(0) for t in targets]
    max_len = max(lengths)
    padded = torch.zeros(len(targets), max_len, dtype=torch.long) 

    for i, t in enumerate(targets):
        padded[i, :lengths[i]] = t

    return images, padded, lengths


In [ ]:

import torch
import torch.nn as nn

 # CNN extracts features from images 
class EncoderCNN(nn.Module):
    def __init__(self, in_channels=1, out_channels=256, input_height=32):
        super().__init__()
        self.out_channels = out_channels
        self.input_height = input_height
         # Feature extractor
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),  

            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),  

            nn.Conv2d(128, out_channels, 3, padding=1), nn.ReLU()
        )

        self.h_reduced = input_height // 4
        self.enc_dim = out_channels * self.h_reduced

    def forward(self, x):
        feat = self.conv(x) 
        B, C, H, W = feat.size()
        # reshape to sequence along width dimension
        enc_out = feat.permute(0, 3, 2, 1).contiguous().view(B, W, H * C)
        return enc_out


class Attention(nn.Module):
    def __init__(self, enc_dim, dec_dim):
        super().__init__()
        self.attn = nn.Linear(enc_dim + dec_dim, dec_dim)
        self.v = nn.Linear(dec_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        h = hidden[0] 
        B, T, _ = encoder_outputs.size()
        h_exp = h.unsqueeze(1).repeat(1, T, 1)  
        energy = torch.tanh(self.attn(torch.cat([h_exp, encoder_outputs], dim=2))) 
        scores = self.v(energy).squeeze(2) 
        attn_weights = torch.softmax(scores, dim=1)  
        context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs).squeeze(1)  
        return context, attn_weights

#Decoder predicts next character one token at a time
class DecoderRNN(nn.Module):
    def __init__(self, vocab_size, enc_dim, dec_dim, embed_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.GRU(embed_dim + enc_dim, dec_dim, batch_first=True)
        self.attention = Attention(enc_dim, dec_dim)
        self.fc = nn.Linear(dec_dim, vocab_size)

    def forward(self, input_char, hidden, encoder_outputs):
    
        emb = self.embedding(input_char).unsqueeze(1)  
        context, attn = self.attention(hidden, encoder_outputs)  
        rnn_in = torch.cat([emb, context.unsqueeze(1)], dim=2)  
        out, hidden = self.rnn(rnn_in, hidden)  
        logits = self.fc(out.squeeze(1))  
        return logits, hidden, attn

#Seq2Seq wrapper connects Encoder + Decoder together
class Seq2Seq(nn.Module):
    def __init__(self, encoder: EncoderCNN, decoder: DecoderRNN, sos_idx: int, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.sos_idx = sos_idx
        self.device = device

    def forward(self, images, targets=None, teacher_forcing_ratio=1.0, max_len=50):
        B = images.size(0)
        enc_out = self.encoder(images.to(self.device))  
        dec_dim = self.decoder.rnn.hidden_size
        if targets is not None:
            T_out = targets.size(1)
        else:
            T_out = max_len

        outputs = torch.zeros(B, T_out, self.decoder.fc.out_features, device=self.device)

        hidden = torch.zeros(1, B, dec_dim, device=self.device)
        input_token = torch.tensor([self.sos_idx] * B, device=self.device, dtype=torch.long)

        for t in range(1, T_out):
            logits, hidden, _ = self.decoder(input_token, hidden, enc_out)  # [B, vocab]
            outputs[:, t] = logits
            if (targets is not None) and (torch.rand(1).item() < teacher_forcing_ratio):
                input_token = targets[:, t].to(self.device)
            else:
                input_token = logits.argmax(1)
        return outputs


In [ ]:

import torch

def greedy_decode(encoder, decoder, image_tensor, idx2char, sos_idx=1, eos_idx=2, max_len=50, device='cpu'):
    # Put encoder and decoder in evaluation mode
    encoder.eval()
    decoder.eval()

    with torch.no_grad():
        # Encode image into sequence representation
        enc_out = encoder(image_tensor.to(device))  
        B = enc_out.size(0)
        assert B == 1, "greedy_decode expects single sample with batch size 1"

        hidden = torch.zeros(1, 1, decoder.rnn.hidden_size, device=device)
         # Start decoding with <SOS>
        input_token = torch.tensor([sos_idx], device=device, dtype=torch.long)
        result_chars = []

        # Loop to predict each next character until EOS
        for _ in range(max_len):
            logits, hidden, _ = decoder(input_token, hidden, enc_out)
            pred = logits.argmax(1).item()
            if pred == eos_idx:
                break
            result_chars.append(idx2char[pred])
            input_token = torch.tensor([pred], device=device, dtype=torch.long)
# Join character list to form final string
    return ''.join(result_chars)


In [ ]:

import torch
import torch.nn as nn
import os

def train(model, dataloader, optimizer, criterion, device, teacher_forcing_ratio=1.0):
    model.train()
    total_loss = 0.0

    for images, targets, lengths in dataloader:
        images = images.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        outputs = model(images, targets, teacher_forcing_ratio=teacher_forcing_ratio)  

        # compute loss
        logits = outputs[:, 1:].reshape(-1, outputs.size(-1))  
        tgt = targets[:, 1:].reshape(-1)                      

        loss = criterion(logits, tgt)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    return avg_loss
#Save models + vocabulary
def save_checkpoint(path, encoder, decoder, vocab):
    state = {
        'encoder': encoder.state_dict(),
        'decoder': decoder.state_dict(),
        'vocab': vocab
    }
    torch.save(state, path)

def load_checkpoint(path, encoder, decoder, device):
    ckpt = torch.load(path, map_location=device)
    encoder.load_state_dict(ckpt['encoder'])
    decoder.load_state_dict(ckpt['decoder'])
    return ckpt.get('vocab', None)


In [ ]:


import torch

def load_image_tensor(image_path, device, size=(32,128)):
    tf = transforms.Compose([
        transforms.Resize(size),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])
    img = Image.open(image_path).convert('L')
    t = tf(img).unsqueeze(0).to(device) 
    return t


In [ ]:

import torch
import os
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd


# ---------- Hyperparams ----------
base_dir = "/content/data_csv/recognition"  
train_csv = os.path.join(base_dir, "train.csv")
checkpoint_path = os.path.join(base_dir, "ocr_seq.pth")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 4
epochs = 10
lr = 1e-3
image_size = (32, 128)  # H, W
in_channels = 1

# ---------- Dataset ----------
transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset = OCRDataset(train_csv, base_dir, transform=transform)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)

vocab = dataset.vocab
char2idx = dataset.char2idx
idx2char = dataset.idx2char
vocab_size = len(vocab)

# ---------- Models ----------
encoder = EncoderCNN(in_channels, out_channels=256, input_height=image_size[0]).to(device)
enc_dim = encoder.enc_dim  # dynamic computed
decoder = DecoderRNN(vocab_size, enc_dim=enc_dim, dec_dim=256, embed_dim=128).to(device)

sos_idx = char2idx['<sos>']
eos_idx = char2idx['<eos>']

seq_model = Seq2Seq(encoder, decoder, sos_idx, device).to(device)

# ---------- Optimizer & Loss ----------
optimizer = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=lr)
criterion = torch.nn.CrossEntropyLoss(ignore_index=0)  # ignore PAD=0

# ---------- Training ----------
for ep in range(1, epochs+1):
    loss = train(seq_model, dataloader, optimizer, criterion, device, teacher_forcing_ratio=1.0)
    print(f"Epoch {ep}/{epochs} Loss: {loss:.4f}")

# ---------- Save checkpoint ----------
save_checkpoint(checkpoint_path, encoder, decoder, vocab)
print("Saved checkpoint:", checkpoint_path)

# ---------- Predict example ----------

_vocab = load_checkpoint(checkpoint_path, encoder, decoder, device)  # returns vocab
if _vocab is not None:
    idx2char = {i:c for i,c in enumerate(_vocab)}

test_image_path = "/content/data_csv/recognition/test/english/A_image_109_10.jpg"  # change as needed
img_t = load_image_tensor(test_image_path, device, size=image_size)

pred_text = greedy_decode(encoder, decoder, img_t, idx2char, sos_idx=sos_idx, eos_idx=eos_idx, max_len=80, device=device)
print("Predicted Text:", pred_text)


Epoch 1/10 Loss: 2.8835
Epoch 2/10 Loss: 2.4203
Epoch 3/10 Loss: 2.2615
Epoch 4/10 Loss: 2.1586
Epoch 5/10 Loss: 2.0882
Epoch 6/10 Loss: 2.0361
Epoch 7/10 Loss: 1.9993
Epoch 8/10 Loss: 1.9750
Epoch 9/10 Loss: 1.9639
Epoch 10/10 Loss: 1.9503
Saved checkpoint: /content/data_csv/recognition/ocr_seq.pth
Predicted Text: OF
